# cli

> The command line: score a file or stdin, warm

In [ ]:
#| default_exp cli

`slopometer <file>` prints the worst-first report, `slopometer` alone reads stdin, and `--threshold` turns the density into an exit code for CI and hooks. Loading the model takes about a second and scoring takes milliseconds, so the command runs through [warmpy](https://github.com/AnswerDotAI/warmpy): the first call starts a background process that loads the model, later calls reuse it, and it exits after thirty idle minutes. The Claude Code stop hook runs this same command with the response text on stdin.

In [ ]:
#| export
import sys
from warmpy import warm_parse

warmpy's contract puts one obligation on this module: it must import fast, because the client process pays this import on every command. So the heavy import happens inside the function body, which only ever runs in the warm background process or in the cold fallback. `warm_parse` mirrors `call_parse`: the docments become the `--help`, and a direct Python call with arguments runs the body in this process, which is what the demo below does.

In [ ]:
from fastcore.test import *
from nbdev.config import get_config

In [ ]:
#| export
@warm_parse
def main(
    path:str=None, # File to score; stdin when omitted
    threshold:float=None, # Exit code 1 when density exceeds this
    json:bool=False, # Emit the result as JSON instead of the report
):
    "Score markdown against the aai reference-prose rules"
    from json import dumps
    from slopometer.house import apply_profile
    apply_profile()  # this fork scores against the house profile unless $SLOPOMETER_PROFILE says otherwise
    from slopometer.score import score_path, score_text
    res = score_path(path) if path else score_text(sys.stdin.read())
    if json: print(dumps(dict(density=res.density, worst=res.worst, words=res.words,
        findings=[{k: getattr(f, k) for k in ('rule', 'tell', 'start', 'end', 'text', 'weight', 'suggestion')} for f in res.findings])))
    else: print(res)
    if threshold is not None and res.density > threshold: return 1


In [ ]:
t2 = get_config().config_path/'samples'/'theory2.md'
test_eq(main(path=str(t2), threshold=5), 1)

/Users/jhoward/aai-ws/slopometer/samples/theory2.md: density 5.4 (weight 19 on 353 prose words), worst 3
1|bbfa| [3] coinage (tell 12, audience misjudged): 'warmpy'
3|1d09| [3] triad (tell 20, forced symmetry): 'directory, same environment, same exit code'
7|066f| [3] passive: 'socket that does not answer is deleted'
7|066f| [3] passive: 'server with the wrong version is replaced'
9|f956| [3] passive: 'life is inferred'
15|3502| [3] rhetorical (tell 18, rhetorical questions): 'The test for every future decision: can the user tell, except by the clock?'
11|6894| [1] noun_cluster: 'mutates module state'


Machine consumers get `--json`: density, worst, prose word count, and every finding's fields, without parsing the report text. The stop hook reads this.

In [ ]:
import json
from io import StringIO
from contextlib import redirect_stdout

In [ ]:
buf = StringIO()
with redirect_stdout(buf): main(path=str(t2), json=True)
j = json.loads(buf.getvalue())
test_eq(set(j) >= {'density', 'worst', 'words', 'findings'}, True)
j['density'], j['findings'][0]

(5.4,
 {'rule': 'coinage',
  'tell': 12,
  'start': 135,
  'end': 141,
  'text': 'warmpy',
  'weight': 3,
  'suggestion': None})

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()